# XGBoost for Cyber Defense
*gradient-boosted trees that ship to the SOC — real detections, real precision, real false-positive budgets*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/ml-course-notebooks/blob/main/xgboost_security_course.ipynb)

18 hands-on episodes for security / detection engineers. Each uses XGBoost on a **seeded, in-code security dataset** (phishing URLs, PE-malware features, NetFlow, DGA, UEBA, DLP, account-takeover, …) — no downloads — and reports the **security-honest metrics** (PR-AUC, precision, recall, FPR @ threshold), not accuracy.

**Every number in this notebook is real:** each section was executed and its printed output captured verbatim, so the code reproduces the figures and headline metrics shown. Run the install cell, then any `## SEC##` section top-to-bottom.

### Episodes
01. Why Trees Win on Telemetry
02. Phishing URLs, End to End
03. The Imbalance Trap
04. Accuracy Lies — Use PR-AUC
05. PE Malware Classifier
06. Network Intrusion on NetFlow
07. Stop Early, Stop Smart
08. DGA Domain Detection
09. Insider Threat / UEBA
10. DLP — Sensitive Data Egress
11. Account Takeover at Login
12. Reading the Why — SHAP for Triage
13. The False-Positive Budget
14. Encoding Threat-Model Priors
15. Calibrating the Risk Score
16. Concept Drift & Retraining
17. Adversarial Evasion
18. The SOC Scoring Pipeline

In [ ]:
!pip install -q xgboost scikit-learn shap matplotlib pandas numpy

## SEC01 · Why Trees Win on Telemetry

On wide, imbalanced security telemetry, your first instinct might be a neural net — resist it. We pit a lone decision tree, an MLP, a random forest, and XGBoost against one identical 28-feature connection-telemetry split (~16% attacks), scored on PR-AUC and a security-honest operating point (precision / recall / false-positive rate at recall ≈ 0.80), not accuracy. Boosted trees win out-of-the-box: `XGBoost PR-AUC 0.780` vs `RandomForest 0.762` · `MLP 0.777` · `DecisionTree 0.461`, and XGBoost also lands the best precision `0.594` and lowest FPR `0.102`.

In [ ]:
# Why gradient-boosted trees win on security telemetry
# Colab: pip install xgboost scikit-learn matplotlib numpy pandas
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve
from xgboost import XGBClassifier

SEED = 7
np.random.seed(SEED)

# --- 1. one seeded world: malicious-vs-benign connections (~16% attacks) ---
# Wide, imbalanced telemetry with sparse signal & noisy interactions —
# the tabular regime where boosted trees beat nets out-of-the-box.
X, y = make_classification(
    n_samples=12000, n_features=28, n_informative=10, n_redundant=5,
    n_clusters_per_class=4, weights=[0.85, 0.15], class_sep=0.8,
    flip_y=0.02, random_state=SEED)
COLS = ["duration", "bytes_in", "bytes_out", "pkts", "syn_ratio",
        "unique_ports", "failed_conns", "tls_version", "geo_risk", "asn_rep",
        "dns_entropy", "ja3_rarity", "beacon_score", "url_len", "ua_rarity",
        "resp_size", "flow_iat", "tcp_flags", "ttl_var", "cert_age",
        "port_scan", "login_fails", "payload_h", "conn_rate",
        "icmp_ratio", "dns_txt", "tls_sni_rare", "http_err_rate"]
X = pd.DataFrame(X, columns=COLS)

# --- 2. ONE stratified split — every model sees identical rows ---
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
# small validation slice carved from train for XGBoost early stopping
Xtr2, Xval, ytr2, yval = train_test_split(
    Xtr, ytr, test_size=0.2, stratify=ytr, random_state=SEED)
pos_w = (ytr == 0).sum() / (ytr == 1).sum()
print(f"attack rate: {y.mean():.1%}  |  scale_pos_weight (benign/attack): {pos_w:.2f}")

# nets need scaling; trees do not — scale a COPY only for the MLP
sc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

# --- 3. four contestants at sane defaults (no tuning) ---
models = {
    "DecisionTree": DecisionTreeClassifier(max_depth=6, random_state=SEED),
    "MLP":          MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=400,
                                  random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED,
                                           n_jobs=-1),
    "XGBoost":      XGBClassifier(n_estimators=800, max_depth=6,
                                  learning_rate=0.05, subsample=0.9,
                                  colsample_bytree=0.8, min_child_weight=2,
                                  reg_lambda=1.5, scale_pos_weight=pos_w,
                                  eval_metric="aucpr", early_stopping_rounds=40,
                                  random_state=SEED),
}

# --- 4. score with PR-AUC on the attack probability (NOT accuracy) ---
print(f"\n{'model':<13} {'PR-AUC':>7} {'prec':>7} {'recall':>7} {'FPR':>7}")
scores = {}
for name, m in models.items():
    if name == "MLP":
        m.fit(Xtr_s, ytr); p = m.predict_proba(Xte_s)[:, 1]
    elif name == "XGBoost":
        # early stopping on a held-out validation slice (real xgboost 3.x API)
        m.fit(Xtr2, ytr2, eval_set=[(Xval, yval)], verbose=False)
        p = m.predict_proba(Xte)[:, 1]
    else:
        m.fit(Xtr, ytr);   p = m.predict_proba(Xte)[:, 1]
    scores[name] = average_precision_score(yte, p)
    # security-honest operating point: threshold for recall >= 0.80
    prec, rec, thr = precision_recall_curve(yte, p)
    ok = np.where(rec[:-1] >= 0.80)[0]
    t = thr[ok[-1]] if len(ok) else 0.5
    pred = (p >= t).astype(int)
    tp = ((pred == 1) & (yte == 1)).sum(); fp = ((pred == 1) & (yte == 0)).sum()
    fn = ((pred == 0) & (yte == 1)).sum(); tn = ((pred == 0) & (yte == 0)).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    print(f"{name:<13} {scores[name]:>7.3f} {precision:>7.3f} {recall:>7.3f} {fpr:>7.3f}")

best = max(scores, key=scores.get)
print(f"\nwinner: {best}  ->  start here, not with a neural net")
print(f"XGBoost beats RandomForest by {scores['XGBoost']-scores['RandomForest']:+.3f} PR-AUC, "
      f"the lone tree by {scores['XGBoost']-scores['DecisionTree']:+.3f}")

# --- 5. hero figure: grouped PR-AUC bars on the same split ---
names = ["DecisionTree", "MLP", "RandomForest", "XGBoost"]
vals  = [scores[n] for n in names]
cols  = ["#79828F", "#79828F", "#FFB454", "#FF5C5C"]
plt.figure(figsize=(7, 4.3)); plt.style.use("dark_background")
bars = plt.bar(names, vals, color=cols, edgecolor="#0A0D12")
for b, v in zip(bars, vals):
    plt.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.3f}",
             ha="center", color="#E8EDF4", fontsize=11)
plt.ylim(0, 1.0); plt.ylabel("test PR-AUC"); plt.title("Same split · zero tuning")
plt.axhline(scores["XGBoost"], color="#FF5C5C", lw=0.8, ls="--", alpha=0.5)
plt.tight_layout(); plt.show()

**Output**

```text
attack rate: 15.7%  |  scale_pos_weight (benign/attack): 5.36

model          PR-AUC    prec  recall     FPR
DecisionTree    0.461   0.244   0.831   0.479
MLP             0.777   0.574   0.801   0.111
RandomForest    0.762   0.536   0.805   0.130
XGBoost         0.780   0.594   0.801   0.102

winner: XGBoost  ->  start here, not with a neural net
XGBoost beats RandomForest by +0.018 PR-AUC, the lone tree by +0.319
```

At a matched recall of ~0.80, XGBoost is the only model that pairs the top PR-AUC with the lowest false-positive rate (0.102) — on wide, imbalanced telemetry, that means fewer analyst-burning false alarms for the same attack coverage.

## SEC02 · Phishing URLs, End to End

We build the recurring **seed-42 'house' detector** end to end: a seeded phishing-vs-benign URL feed with real domain feature names, a single plain XGBoost baseline, and a confusion matrix at the default 0.5 threshold. We score the security-honest way — precision, recall, and false-positive rate — and show *why accuracy lies* on an 80/20 imbalanced feed.

Real results on the seeded test set: `PR-AUC 0.958 · precision 0.95 · recall 0.85 · FPR 1.17%` (accuracy `0.96` vs a do-nothing baseline of `0.80`).

In [ ]:
# SEC02 — Phishing URLs, end to end. The seed-42 'house' detector.
# Honest baseline: PR-AUC, precision, recall, FPR (never accuracy alone).
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             accuracy_score)
import matplotlib.pyplot as plt
import xgboost as xgb

FEATURES = ["url_length", "num_dots", "has_https", "subdomain_count",
            "entropy", "age_days", "num_hyphens", "has_at_symbol",
            "path_depth", "num_digits", "brand_in_subdomain", "redirect_count"]

# Canonical seed-42 'house' dataset reused across SEC02/04/12/13/15/18.
X, y = make_classification(
    n_samples=12000, n_features=12, n_informative=8,
    weights=[0.8, 0.2], class_sep=1.1, random_state=42)
X = pd.DataFrame(X, columns=FEATURES)

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

clf = xgb.XGBClassifier(
    n_estimators=450, max_depth=7, learning_rate=0.12,
    subsample=0.9, colsample_bytree=0.9,
    eval_metric="aucpr", random_state=42, n_jobs=4)
clf.fit(Xtr, ytr)

scores = clf.predict_proba(Xte)[:, 1]
pr_auc = average_precision_score(yte, scores)

pred = (scores >= 0.50).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
precision = tp / (tp + fp)
recall    = tp / (tp + fn)
fpr       = fp / (fp + tn)

print(f"test set: {tp+fn} phishing, {tn+fp} benign  (n={len(yte)})")
print(f"confusion matrix  tn={tn} fp={fp} fn={fn} tp={tp}")
print(f"PR-AUC={pr_auc:.3f}  precision={precision:.2f}  "
      f"recall={recall:.2f}  FPR={fpr*100:.2f}%  (thr 0.50)")

acc = accuracy_score(yte, pred)
base = max((yte == 0).mean(), (yte == 1).mean())
print(f"accuracy={acc:.2f}  (do-nothing baseline={base:.2f}) <- why accuracy lies")

# Hero figure: confusion matrix at threshold 0.5
cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["benign", "phishing"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["benign", "phishing"])
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
ax.set_title("Phishing detector — confusion matrix (thr 0.50)")
fig.tight_layout()
plt.show()

**Output**

```text
test set: 611 phishing, 2389 benign  (n=3000)
confusion matrix  tn=2361 fp=28 fn=93 tp=518
PR-AUC=0.958  precision=0.95  recall=0.85  FPR=1.17%  (thr 0.50)
accuracy=0.96  (do-nothing baseline=0.80) <- why accuracy lies
```

## SEC03 · The Imbalance Trap

**The imbalance trap.** Only **1.5%** of NetFlow records are ransomware beacons. Train XGBoost normally and the 0.5 threshold quietly ignores the minority class — it misses **more than half** the attacks while reporting near-perfect precision. The fix is one knob computed from the data: `scale_pos_weight = neg/pos`. It rebalances the gradient so the model pays attention to the rare positives.

Real seeded run (`scale_pos_weight ≈ 51`): attack **recall `0.44 → 0.59`** (beacons caught `43 → 57` of 97), **precision `0.93 → 0.55`**, **PR-AUC `0.63 → 0.59`**. Recall climbs sharply; precision is the price you pay; PR-AUC stays flat because the knob shifts the *threshold*, not the *ranking*. Accuracy would have looked great the whole time — which is exactly the trap.

In [ ]:
# SEC03 — The Imbalance Trap: default vs scale_pos_weight = neg/pos
# Ransomware-beacon detection on NetFlow. ~1.5% of flows are attacks.
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (recall_score, precision_score,
                             average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay)
import xgboost as xgb

# --- 1. Seeded world: 20k flows, ~1.5% ransomware beacons ----------------
FEATURES = ["beacon_interval_var", "bytes_per_flow", "jitter",
            "dst_port_entropy", "conn_count", "tls_ja3_rare",
            "dns_txt_ratio", "ua_entropy", "off_hours_ratio",
            "c2_asn_rep", "sni_entropy"]
# class_sep=1.3 + 9 informative feats: beacons ARE detectable, but the
# default 0.5 threshold still ignores the 1.5% minority. The real trap.
X, y = make_classification(n_samples=20000, n_features=11, n_informative=9,
                           n_redundant=0, n_repeated=0,
                           weights=[0.985, 0.015], class_sep=1.3,
                           random_state=11)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=11)

# --- 2. The knob, computed from data (never hard-coded) ------------------
spw = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f"train: {(y_tr==1).sum()} beacons / {(y_tr==0).sum()} benign  "
      f"-> scale_pos_weight = {spw:.0f}")

# --- 3. Same model twice; ONLY scale_pos_weight differs ------------------
def fit(weight):
    m = xgb.XGBClassifier(n_estimators=300, max_depth=4, eta=0.1,
                          subsample=0.9, colsample_bytree=0.9,
                          eval_metric="aucpr", scale_pos_weight=weight,
                          random_state=11, n_jobs=4)
    return m.fit(X_tr, y_tr)

m_default  = fit(1.0)
m_weighted = fit(spw)

# --- 4. Honest, imbalance-aware metrics (accuracy lies here) -------------
def report(model, name):
    p = model.predict(X_te)                  # labels at threshold 0.5
    s = model.predict_proba(X_te)[:, 1]      # scores for PR-AUC
    cm = confusion_matrix(y_te, p)
    tn, fp, fn, tp = cm.ravel()
    return dict(name=name, recall=recall_score(y_te, p),
                precision=precision_score(y_te, p),
                prauc=average_precision_score(y_te, s),
                fpr=fp / (fp + tn), cm=cm, fn=fn, tp=tp)

r0 = report(m_default,  "default (spw=1)")
r1 = report(m_weighted, f"weighted (spw={spw:.0f})")

print(f"recall    {r0['recall']:.2f} -> {r1['recall']:.2f}")
print(f"precision {r0['precision']:.2f} -> {r1['precision']:.2f}")
print(f"PR-AUC    {r0['prauc']:.2f} -> {r1['prauc']:.2f}")
print(f"FPR@0.5   {r0['fpr']:.3f} -> {r1['fpr']:.3f}")
print(f"beacons caught (TP)  {r0['tp']} -> {r1['tp']}  of {r0['tp']+r0['fn']}")
print(f"beacons missed (FN)  {r0['fn']} -> {r1['fn']}")

# --- 5. Hero figure: two confusion matrices side by side ----------------
fig, ax = plt.subplots(1, 2, figsize=(10, 4.4))
for a, r in zip(ax, [r0, r1]):
    ConfusionMatrixDisplay(r["cm"], display_labels=["benign", "beacon"]).plot(
        ax=a, cmap="Reds", colorbar=False, values_format="d")
    a.set_title(f"{r['name']}\nrecall={r['recall']:.2f}  PR-AUC={r['prauc']:.2f}")
fig.suptitle(f"The Imbalance Trap — missed beacons drop {r0['fn']} -> {r1['fn']}")
fig.tight_layout()
plt.show()

**Output**

```text
train: 289 beacons / 14711 benign  -> scale_pos_weight = 51
recall    0.44 -> 0.59
precision 0.93 -> 0.55
PR-AUC    0.63 -> 0.59
FPR@0.5   0.001 -> 0.010
beacons caught (TP)  43 -> 57  of 97
beacons missed (FN)  54 -> 40
```

With `scale_pos_weight = 51`, attack recall jumps `0.44 → 0.59` — 14 more beacons caught (43 → 57 of 97), and missed beacons fall 54 → 40. Precision pays for it (`0.93 → 0.55`) and the false-positive rate ticks up `0.1% → 1.0%`. PR-AUC is essentially flat (`0.63 → 0.59`) because the knob moves the *decision threshold*, not the model's *ranking* of flows. In cyber defense a missed beacon is a breach, so trading some precision for recall is usually the right call — but tune the knob to your alert budget, don't trust it blindly.

## SEC04 · Accuracy Lies — Use PR-AUC

A do-nothing model that flags *every* site as benign scores **80% accuracy** on this 20%-phishing dataset — and catches zero attacks. Accuracy rewards it for the class imbalance alone. PR-AUC vs the prevalence floor is the only honest headline: it pins the no-skill baseline at the positive rate, so a useless model scores there and a real one has to climb above it.

Real run: all-benign `acc 0.797 · PR-AUC 0.203` vs XGBoost `acc 0.960 · PR-AUC 0.958` (prevalence floor `0.203`).

In [ ]:
# SEC04 — Accuracy Lies, Use PR-AUC
# The phishing detector vs a do-nothing "all-benign" model.
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, PrecisionRecallDisplay,
    precision_score, recall_score, confusion_matrix,
)
from xgboost import XGBClassifier

SEED = 42  # the recurring seed-42 phishing 'house' dataset

# 1) Rebuild the house: ~20% phishing, ~80% benign — imbalanced like the real web
X, y = make_classification(
    n_samples=12000, n_features=12, n_informative=8,
    weights=[0.8, 0.2], class_sep=1.1, random_state=SEED,
)
FEATURES = ["url_length", "num_dots", "has_https", "subdomain_count",
            "entropy", "age_days", "num_params", "path_depth",
            "has_at_symbol", "tld_rank", "redirect_count", "form_count"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
prevalence = yte.mean()  # positive (phishing) fraction == the PR-AUC no-skill floor

# 2) The villain: predict the majority class (benign) for everyone. Learns nothing.
dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
p_dummy = dummy.predict_proba(Xte)[:, 1]      # flat constant -> flat PR curve

# 3) The real detector: XGBoost with early stopping
xgb = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9,
    eval_metric="aucpr", early_stopping_rounds=30, random_state=SEED,
)
xgb.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
p_xgb = xgb.predict_proba(Xte)[:, 1]
yhat_xgb = (p_xgb >= 0.5).astype(int)

# 4) The reveal: accuracy says they're close; PR-AUC says one is furniture
acc_dummy = accuracy_score(yte, dummy.predict(Xte))
acc_xgb   = accuracy_score(yte, yhat_xgb)
ap_dummy  = average_precision_score(yte, p_dummy)   # == prevalence
ap_xgb    = average_precision_score(yte, p_xgb)

# Security-honest operating-point metrics for XGBoost @ threshold 0.5
prec = precision_score(yte, yhat_xgb)
rec  = recall_score(yte, yhat_xgb)
tn, fp, fn, tp = confusion_matrix(yte, yhat_xgb).ravel()
fpr = fp / (fp + tn)  # benign sites wrongly flagged

print("=== Accuracy lies, PR-AUC tells the truth ===")
print(f"prevalence (PR-AUC no-skill floor) = {prevalence:.3f}")
print(f"AllBenign  acc={acc_dummy:.3f}  PR-AUC={ap_dummy:.3f}  (catches 0 attacks)")
print(f"XGBoost    acc={acc_xgb:.3f}  PR-AUC={ap_xgb:.3f}")
print(f"XGBoost @0.5  precision={prec:.3f}  recall={rec:.3f}  FPR={fpr:.3f}")
print(f"  confusion: TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"accuracy gap = {acc_xgb-acc_dummy:.3f}  |  PR-AUC gap = {ap_xgb-ap_dummy:.3f}")

# 5) Plot both PR curves with the prevalence floor; annotate accuracy on each
fig, ax = plt.subplots(figsize=(7, 6))
PrecisionRecallDisplay.from_predictions(yte, p_xgb, ax=ax,
    name=f"XGBoost (acc {acc_xgb:.2f}, PR-AUC {ap_xgb:.2f})",
    curve_kwargs={"color": "#56D364"})
PrecisionRecallDisplay.from_predictions(yte, p_dummy, ax=ax,
    name=f"All-benign (acc {acc_dummy:.2f}, PR-AUC {ap_dummy:.2f})",
    curve_kwargs={"color": "#FF5C5C"})
ax.axhline(prevalence, ls="--", color="#FFB454", label=f"prevalence floor = {prevalence:.2f}")
ax.set_title("Accuracy lies — PR-AUC tells the truth")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

**Output**

```text
=== Accuracy lies, PR-AUC tells the truth ===
prevalence (PR-AUC no-skill floor) = 0.203
AllBenign  acc=0.797  PR-AUC=0.203  (catches 0 attacks)
XGBoost    acc=0.960  PR-AUC=0.958
XGBoost @0.5  precision=0.950  recall=0.850  FPR=0.012
  confusion: TP=415 FP=22 FN=73 TN=1890
accuracy gap = 0.164  |  PR-AUC gap = 0.755
```

The all-benign model hits 80% accuracy and catches nothing — its PR-AUC sits *exactly* on the 0.203 prevalence floor. XGBoost's accuracy edge looks modest (+0.16), but the PR-AUC gap is +0.755: that's the real story accuracy hides.

## SEC05 · PE Malware Classifier

Static PE-malware detection from header features — no binaries downloaded, just a seeded synthetic world where the strongest signals map onto real malware tells. We train XGBoost with early stopping and read it the way a SOC analyst would: not by accuracy, but by recall at a fixed false-positive budget. The hero is the **gain** importance chart — it tells you *which* structural features the trees lean on, but not in which direction.

Real run: `PR-AUC 0.985` · `recall 0.896 @ FPR 1.2% (thr 0.77)` · `precision 0.980` · top gain `suspicious_api_count · packer_score · entropy`.

In [ ]:
# SEC05 · PE Malware Classifier — static detection from PE-header features
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, confusion_matrix
import xgboost as xgb
import matplotlib.pyplot as plt

SEED = 23
np.random.seed(SEED)

# --- 1 · seeded PE-header world (no downloads) -------------------------------
X, y = make_classification(
    n_samples=15000, n_features=12, n_informative=9,
    weights=[0.60, 0.40], class_sep=1.2, random_state=SEED,
)
# Map synthetic columns onto real PE-header names so the strongest
# structural signals carry the genuine malware tells.
RAW2PE = {
    "f8": "suspicious_api_count", "f2": "entropy",       "f1": "is_signed",
    "f6": "packer_score",         "f10": "n_imports",    "f5": "has_tls_section",
    "f7": "overlay_size",         "f4": "resource_size", "f3": "n_sections",
    "f9": "debug_present",        "f0": "n_exports",     "f11": "timestamp_anomaly",
}
df = pd.DataFrame(X, columns=[f"f{i}" for i in range(12)]).rename(columns=RAW2PE)
FEATURES = list(RAW2PE.values())
df = df[FEATURES]                      # 1 = malware, 0 = benign

# --- 2 · stratified split (frozen) ------------------------------------------
Xtr, Xte, ytr, yte = train_test_split(
    df, y, test_size=0.25, stratify=y, random_state=SEED)

# --- 3 · train XGBoost (early stopping on held-out eval set) -----------------
clf = xgb.XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=1.0,
    eval_metric="aucpr", early_stopping_rounds=30,
    random_state=SEED, n_jobs=4)
clf.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)

# --- 4 · security-honest metrics --------------------------------------------
proba = clf.predict_proba(Xte)[:, 1]
prauc = average_precision_score(yte, proba)
best = None                            # pin FPR at the 1.2% blocklist budget
for t in np.linspace(0.05, 0.99, 400):
    tn, fp, fn, tp = confusion_matrix(yte, (proba >= t).astype(int)).ravel()
    fpr, rec = fp / (fp + tn), tp / (tp + fn)
    if best is None or abs(fpr - 0.012) < abs(best[2] - 0.012):
        best = (t, rec, fpr)
thr, recall, fpr = best
tn, fp, fn, tp = confusion_matrix(yte, (proba >= thr).astype(int)).ravel()
prec = tp / (tp + fp)
print(f"best iteration    : {clf.best_iteration}")
print(f"PR-AUC            : {prauc:.3f}")
print(f"recall @ FPR {fpr*100:.1f}% : {recall:.3f}  (threshold {thr:.2f})")
print(f"precision @ thr   : {prec:.3f}")

# --- 5 · gain importance — the hero figure ----------------------------------
gain = clf.get_booster().get_score(importance_type="gain")
gain = sorted(gain.items(), key=lambda kv: kv[1], reverse=True)[:12]
names, vals = [g[0] for g in gain], [g[1] for g in gain]
print("top gain (3)      :", ", ".join(names[:3]))
print("gain values (top5):", ", ".join(f"{n}={v:.1f}" for n, v in gain[:5]))

plt.figure(figsize=(8, 6))
plt.barh(names[::-1], vals[::-1], color="#FF5C5C")
plt.xlabel("gain"); plt.title("PE malware — feature importance (gain)")
plt.tight_layout(); plt.show()

**Output**

```text
best iteration    : 193
PR-AUC            : 0.985
recall @ FPR 1.2% : 0.896  (threshold 0.77)
precision @ thr   : 0.980
top gain (3)      : suspicious_api_count, packer_score, entropy
gain values (top5): suspicious_api_count=130.5, packer_score=56.4, entropy=55.4, is_signed=49.6, n_imports=45.7
```

`suspicious_api_count` dominates by gain (130.5 — more than double the next feature), with `packer_score` and `entropy` behind it: gain surfaces the structural tells the trees actually split on. But note what it does **not** tell you — direction. A high gain on `is_signed` doesn't say whether signed pushes toward malware or benign; for that you need SHAP. And the honest operating point matters: at a strict 1.2% false-positive budget the classifier catches 89.6% of malware (recall), not 95% — the gap is the cost of keeping analysts from drowning in false alarms.

## SEC06 · Network Intrusion on NetFlow

Tune the four tree knobs (`max_depth`, `learning_rate`, `subsample`, `colsample_bytree`) on a seeded NetFlow IDS, log **both** train and validation logloss every boosting round, and let early stopping pick where validation bottoms out. The hero figure is the train-vs-validation learning curve: the gap between the two lines is your overfit gauge. Real run: `best_iter=227 · train_ll 0.015 · val_ll 0.087 · overfit gap 0.071 · PR-AUC 0.983 · recall 0.92 @ FPR 0.9%`.

In [ ]:
# 06_netflow_ids.py — tune the tree knobs, read the learning curve
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_curve
import xgboost as xgb

SEED = 5
FEATURES = ["duration","bytes_in","bytes_out","pkts","syn_ratio",
            "fin_ratio","dst_port_entropy","flow_iat_mean",
            "src_bytes_ratio","tcp_flags_var","pkt_size_var",
            "conn_state","land_flag","urg_count"]

# ── seeded NetFlow world: 18k connections, ~25% hostile ──
X, y = make_classification(
    n_samples=18000, n_features=14, n_informative=10,
    weights=[0.75, 0.25], class_sep=1.0, random_state=SEED)
# X columns ARE the flow features above (renamed, same array)

# ── stratified split keeps the attack ratio in both halves ──
Xtr, Xval, ytr, yval = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)

# ── one model, the four knobs this episode is about ──
clf = xgb.XGBClassifier(
    max_depth=6,            # capacity — shallow generalizes
    learning_rate=0.1,      # eta — small steps, smooth descent
    subsample=0.8,          # 80% of flows per tree (row noise)
    colsample_bytree=0.8,   # 80% of features per tree (col noise)
    n_estimators=600,       # ceiling; early stopping picks the real number
    eval_metric="logloss",
    early_stopping_rounds=40,
    random_state=SEED, n_jobs=2)

# ── eval_set logs BOTH curves every round → the learning curve ──
clf.fit(Xtr, ytr,
        eval_set=[(Xtr, ytr), (Xval, yval)],
        verbose=False)

# ── pull the per-round logloss out of the model ──
res = clf.evals_result()
train_ll = res["validation_0"]["logloss"]   # train curve
val_ll   = res["validation_1"]["logloss"]   # validation curve
best = clf.best_iteration                   # round val bottomed
print(f"depth=6 eta=0.1 -> best_iter={best} | "
      f"train_ll {train_ll[best]:.3f} | val_ll {val_ll[best]:.3f}")
print(f"overfit gap (val-train) = {val_ll[best]-train_ll[best]:.3f}")

# ── SOC-honest metrics (accuracy lies) ──
p = clf.predict_proba(Xval)[:, 1]
prauc = average_precision_score(yval, p)
fpr, tpr, thr = roc_curve(yval, p)
i = int(np.argmin(np.abs(fpr - 0.009)))     # recall at 0.9% FPR
print(f"PR-AUC {prauc:.3f} | recall {tpr[i]:.2f} @ FPR {fpr[i]*100:.1f}%")

# ── figure: train vs validation learning curve ──
import matplotlib.pyplot as plt
rounds = range(len(train_ll))
plt.figure(figsize=(9, 9))
plt.plot(rounds, train_ll, color="#56D364", lw=3, label="train")
plt.plot(rounds, val_ll,   color="#FF5C5C", lw=3, label="validation")
plt.axvline(best, color="#FFB454", lw=2, ls="--",
            label=f"best_iter = {best}")
plt.xlabel("boosting round"); plt.ylabel("logloss"); plt.legend()
plt.show()

**Output**

```text
depth=6 eta=0.1 -> best_iter=227 | train_ll 0.015 | val_ll 0.087
overfit gap (val-train) = 0.071
PR-AUC 0.983 | recall 0.92 @ FPR 0.9%
```

Validation logloss bottoms at round 227, where early stopping freezes the model. Train logloss has collapsed to 0.015 while validation holds at 0.087 — that 0.071 gap is the memorization tax. PR-AUC 0.983 and 92% recall at a 0.9% false-positive rate mean the analyst chases roughly one false alarm per hundred benign flows to catch 92% of intrusions.

## SEC07 · Stop Early, Stop Smart

### Stop Early, Stop Smart

You hand XGBoost a budget of 2000 trees — but a budget is a *ceiling*, not a target. With a held-out validation set watched by `early_stopping_rounds`, the model trains until validation logloss stops improving, then keeps the trough as `best_iteration`. Fewer trees, better generalization, no overfitting tax.

Here the validation curve bottoms out and training halts on its own: stopped at round `149/2000` (built `150` trees, saved `1850`), with an honest test scoreboard of val logloss `0.124` · PR-AUC `0.974` · recall `0.908` · FPR@0.5 `0.022`.

In [ ]:
# Stop Early, Stop Smart: let validation pick best_iteration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, precision_score, recall_score
import xgboost as xgb

# 1) One seeded world: 16k emails, 8 of 13 features informative, 70/30 benign:spam
X, y = make_classification(
    n_samples=16000, n_features=13, n_informative=8,
    weights=[0.7, 0.3], class_sep=1.0, random_state=31,
)
cols = ["n_links", "n_recipients", "reply_to_mismatch", "urgency_kw_hits",
        "money_kw_hits", "spf_fail", "dkim_fail", "domain_age_days",
        "attachment_count", "html_ratio", "sender_rep", "subject_entropy",
        "lookalike_domain"]
df = pd.DataFrame(X, columns=cols)

# 2) THREE-way split: train / validation (watched by early stopping) / test (honest)
X_tr, X_te, y_tr, y_te = train_test_split(
    df, y, test_size=0.25, stratify=y, random_state=31)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_tr, y_tr, test_size=0.20, stratify=y_tr, random_state=31)

# 3) Budget 2000 rounds — a CEILING, not a target. logloss is the early-stop metric.
clf = xgb.XGBClassifier(
    n_estimators=2000, max_depth=3, learning_rate=0.3,
    subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
    gamma=1.0, reg_alpha=1.0, eval_metric="logloss",
    early_stopping_rounds=12, random_state=31, n_jobs=2,
)

# 4) Fit with an eval_set — XGBoost scores it every round and stops at the trough
clf.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)

best = clf.best_iteration
curve = clf.evals_result()["validation_0"]["logloss"]
print(f"Stopped at round {best} / 2000  (built {best + 1} trees, "
      f"saved {2000 - best - 1})")
print(f"val logloss @ best : {curve[best]:.3f}")

# 5) Honest scoreboard on the UNTOUCHED test set
proba = clf.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)
tn = int(((pred == 0) & (y_te == 0)).sum())
fp = int(((pred == 1) & (y_te == 0)).sum())
fpr = fp / (fp + tn)
print(f"PR-AUC   : {average_precision_score(y_te, proba):.3f}")
print(f"precision: {precision_score(y_te, pred):.3f}")
print(f"recall   : {recall_score(y_te, pred):.3f}")
print(f"FPR@0.5  : {fpr:.3f}")

# 6) Hero figure: validation logloss vs round, amber line on best_iteration
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(curve)), curve, color="#FF5C5C", lw=2, label="validation logloss")
ax.axvline(best, color="#FFB454", lw=2, ls="--", label=f"best_iteration = {best}")
ax.set_xlabel("boosting round"); ax.set_ylabel("validation logloss")
ax.set_title("Early stopping — the trough is the model")
ax.legend(); fig.tight_layout()
plt.show()

**Output**

```text
Stopped at round 149 / 2000  (built 150 trees, saved 1850)
val logloss @ best : 0.124
PR-AUC   : 0.974
precision: 0.947
recall   : 0.908
FPR@0.5  : 0.022
```

## SEC08 · DGA Domain Detection

DGA (domain-generation-algorithm) names are *statistically loud* — high entropy, weird n-grams, no dictionary words. That makes them highly separable, so XGBoost can run at a brutally tight false-positive rate and still catch nearly everything, which is exactly what an auto-blocklist needs. Read recall at a fixed FPR, never accuracy. Real run: `ROC-AUC 0.990 · PR-AUC 0.984 · recall 0.94 @ FPR 0.45% (block above score 0.377)`.

In [ ]:
# DGA domain detection: catch algorithmically-generated C2 domains.
# Lesson: lexical/statistical features make DGA highly separable; read recall at a STRICT FPR.
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import xgboost as xgb

# --- one seeded world: 14k domains, 10 lexical features, ~18% DGA -------------
FEATURES = [
    "domain_length", "shannon_entropy", "vowel_ratio", "digit_ratio",
    "ngram_rarity", "consonant_run_max", "tld_rank", "hyphen_count",
    "dict_word_ratio", "reg_age_days",
]
X, y = make_classification(
    n_samples=14000, n_features=10, n_informative=7, n_redundant=2,
    weights=[0.82, 0.18],   # DGA is the 18% minority — rare but separable
    class_sep=2.0,          # human vs machine names pull genuinely apart
    flip_y=0.006,           # a little label noise so the model can't cheat
    random_state=17,
)
df = pd.DataFrame(X, columns=FEATURES)
df["is_dga"] = y            # 1 = algorithmically-generated C2, 0 = human-registered

# --- stratified split keeps the 18% DGA share in both halves ------------------
Xtr, Xte, ytr, yte = train_test_split(
    df[FEATURES], df["is_dga"],
    test_size=0.25, stratify=df["is_dga"], random_state=17,
)

# --- plain XGBoost: depth + eta are the only real knobs here ------------------
clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.8,
    eval_metric="auc", random_state=17, n_jobs=4,
)
clf.fit(Xtr, ytr)
proba = clf.predict_proba(Xte)[:, 1]

# --- security-honest metrics: ROC-AUC + PR-AUC, never accuracy -----------------
roc_auc = roc_auc_score(yte, proba)
pr_auc = average_precision_score(yte, proba)

# --- the whole trade-off curve, then read ONE operating point -----------------
fpr, tpr, thr = roc_curve(yte, proba)     # tpr == recall
TARGET_FPR = 0.005                        # blocklist ceiling: 0.5% false positives
i = np.searchsorted(fpr, TARGET_FPR, side="right") - 1
recall_at = tpr[i]
fpr_at = fpr[i]
thr_at = thr[i]                           # the score cutoff that SHIPS (not 0.5)

print(f"ROC-AUC          : {roc_auc:.3f}")
print(f"PR-AUC           : {pr_auc:.3f}")
print(f"FPR ceiling      : {TARGET_FPR*100:.1f}%   (actual {fpr_at*100:.2f}%)")
print(f"recall @ FPR<=0.5%: {recall_at:.2f}    (block above score {thr_at:.3f})")

# --- hero figure: ROC curve, operating point annotated ------------------------
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "--", color="#1b212b", lw=1)         # random baseline
ax.plot(fpr, tpr, color="#7cc1ff", lw=2.5, label=f"ROC (AUC={roc_auc:.3f})")
ax.axvline(TARGET_FPR, color="#FFB454", lw=1.5)              # the 0.5% FPR rule
ax.scatter([fpr_at], [recall_at], color="#FF5C5C", s=80, zorder=5)
ax.annotate(f"recall {recall_at:.2f} @ FPR 0.5%",
            (fpr_at, recall_at), xytext=(0.12, 0.80),
            color="#FF5C5C",
            arrowprops=dict(arrowstyle="->", color="#FF5C5C"))
ax.set_xlim(0, 0.10); ax.set_ylim(0, 1.01)                  # zoom the low-FPR wall
ax.set_xlabel("false positive rate"); ax.set_ylabel("recall")
ax.legend(loc="lower right")
plt.show()

**Output**

```text
ROC-AUC          : 0.990
PR-AUC           : 0.984
FPR ceiling      : 0.5%   (actual 0.45%)
recall @ FPR<=0.5%: 0.94    (block above score 0.377)
```

The ROC curve hugs the top-left wall. At the shipping operating point — block any domain scoring above **0.377** — the model catches **94%** of DGA traffic while flagging only **0.45%** of legitimate domains, comfortably under the 0.5% auto-blocklist ceiling.

## SEC09 · Insider Threat / UEBA

User and Entity Behavior Analytics (UEBA) flags insider threats from behavioral signals — but a score nobody can explain gets ignored by the SOC. We train XGBoost on a rare-class (3% insider) behavioral dataset, then use SHAP to make every risk score auditable. On a held-out set the model hits `PR-AUC 0.717` (vs `0.035` random) and `recall 0.77 @ FPR ~2%`, with the top global drivers being `vpn_geo_changes`, `logins_per_hr`, and `repo_clones`.

In [ ]:
import numpy as np
import shap
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_curve

SEED = 29
FEATURES = [
    "logins_per_hr", "off_hour_ratio", "distinct_hosts", "data_egress_mb",
    "usb_events", "failed_auth", "priv_esc_attempts", "vpn_geo_changes",
    "after_hours_files", "repo_clones", "badge_anomaly",
]

# seeded UEBA world: ~3% insiders, 11 behavioral features
X, y = make_classification(
    n_samples=10_000, n_features=11, n_informative=7, n_redundant=2,
    weights=[0.97, 0.03], class_sep=0.85, flip_y=0.01, random_state=SEED,
)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED,
)

# XGBoost, gradient rebalanced toward the rare insider class
spw = (ytr == 0).sum() / (ytr == 1).sum()
model = xgb.XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    scale_pos_weight=spw, eval_metric="aucpr",
    random_state=SEED, n_jobs=4,
)
model.fit(Xtr, ytr)

# security-honest scorecard (NOT accuracy)
proba = model.predict_proba(Xte)[:, 1]
base_rate = yte.mean()
pr_auc = average_precision_score(yte, proba)
fpr, tpr, thr = roc_curve(yte, proba)
i = np.argmin(np.abs(fpr - 0.02))
print(f"insiders in test      : {int(yte.sum())}/{len(yte)} ({base_rate*100:.1f}% base rate)")
print(f"PR-AUC                : {pr_auc:.3f}  (vs {base_rate:.3f} random)")
print(f"recall @ FPR {fpr[i]*100:4.1f}% : {tpr[i]:.2f}  (threshold {thr[i]:.3f})")

# SHAP: explain the score globally (TreeSHAP)
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(Xte)
mean_abs = np.abs(sv).mean(axis=0)
order = np.argsort(mean_abs)[::-1]
print("\ntop behavioral drivers (mean |SHAP|):")
for j in order[:3]:
    print(f"  {FEATURES[j]:<18}: {mean_abs[j]:.3f}")

# the hero figure: global beeswarm
shap.summary_plot(sv, Xte, feature_names=FEATURES, plot_type="dot", show=True)

**Output**

```text
insiders in test      : 88/2500 (3.5% base rate)
PR-AUC                : 0.717  (vs 0.035 random)
recall @ FPR  1.7% : 0.77  (threshold 0.533)

top behavioral drivers (mean |SHAP|):
  vpn_geo_changes   : 1.300
  logins_per_hr     : 0.967
  repo_clones       : 0.791
```

PR-AUC of `0.717` against a `3.5%` base rate is roughly 20x better than random, and the model recovers `77%` of insiders while keeping the false-positive rate near `2%` — a workable SOC alert budget. The SHAP beeswarm makes each score auditable: the analyst sees exactly which behaviors pushed a user into the high-risk band.

## SEC10 · DLP — Sensitive Data Egress

DLP isn't graded on accuracy — it's graded on *precision*. Every false alarm is a security analyst's wasted hour, so we don't accept the model's default 0.5 cutoff: we walk the precision-recall curve and pick the threshold where **precision crosses 0.95**, then read off the recall and false-positive rate we can live with. No `scale_pos_weight`, no resampling — just an honest curve and a deliberate operating point.

Real run: `PR-AUC 0.950` · at `precision 0.952` → `recall 0.93` · `FPR 0.5%` (threshold `0.162`).

In [ ]:
# DLP egress detector tuned to a high-precision operating point
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, precision_recall_curve, confusion_matrix

SEED = 44

# --- 1. Seeded, in-code DLP egress world (no downloads) ---
X, y = make_classification(
    n_samples=13000, n_features=10, n_informative=7,
    weights=[0.9, 0.1], class_sep=1.15, random_state=SEED,
)
FEATURES = ["pct_digits", "has_ssn_pattern", "has_cc_pattern", "kw_hits",
            "doc_entropy", "file_size_kb", "recipient_external",
            "encrypted_flag", "secret_regex_hits", "label_confidential"]
df = pd.DataFrame(X, columns=FEATURES)          # y=1 -> sensitive egress (~10%)

X_tr, X_te, y_tr, y_te = train_test_split(
    df, y, test_size=0.25, stratify=y, random_state=SEED)

# --- 2. Plain XGBoost — deliberately NO scale_pos_weight (DLP tunes the threshold) ---
clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    eval_metric="aucpr", random_state=SEED, n_jobs=4,
)
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

# --- 3. Honest scoreboard: PR-AUC over the whole curve ---
pr_auc = average_precision_score(y_te, proba)
precision, recall, thresh = precision_recall_curve(y_te, proba)

# --- 4. THE DLP MOVE: walk the curve to the precision>=0.95 operating point ---
TARGET_PRECISION = 0.95
ok = precision[:-1] >= TARGET_PRECISION          # align with thresh (len-1)
best = np.where(ok)[0][np.argmax(recall[:-1][ok])]   # highest recall that still keeps precision
op_thresh = thresh[best]
op_precision, op_recall = precision[best], recall[best]

# --- 5. Confirm precision/recall/FPR from the confusion matrix at that threshold ---
pred = (proba >= op_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
fpr = fp / (fp + tn)
print(f"PR-AUC {pr_auc:.3f} | @precision>=0.95: recall {op_recall:.2f}, FPR {fpr*100:.1f}%")
print(f"  threshold={op_thresh:.3f}  precision={op_precision:.3f}  "
      f"tp={tp} fp={fp} fn={fn} tn={tn}")

# --- 6. Hero figure: PR curve + amber budget line + marked operating point ---
plt.figure(figsize=(7, 6))
plt.plot(recall, precision, color="#FF5C5C", lw=2.5, label="PR curve")
plt.axhline(TARGET_PRECISION, color="#FFB454", lw=1.5, ls="--", label="precision = 0.95")
plt.scatter([op_recall], [op_precision], color="#56D364", s=90, zorder=5)
plt.annotate(f"recall {op_recall:.2f}\nFPR {fpr*100:.1f}%",
             (op_recall, op_precision), xytext=(op_recall-0.34, op_precision-0.18),
             color="#56D364", fontsize=11)
plt.xlabel("recall"); plt.ylabel("precision")
plt.title(f"DLP egress — PR-AUC {pr_auc:.3f}")
plt.xlim(0, 1); plt.ylim(0, 1.02); plt.legend(loc="lower left")
plt.tight_layout(); plt.show()

**Output**

```text
PR-AUC 0.950 | @precision>=0.95: recall 0.93, FPR 0.5%
  threshold=0.162  precision=0.952  tp=316 fp=16 fn=24 tn=2894
```

The model never reaches `precision = 1.0`, but it crosses 0.95 with room to spare. At threshold `0.162` we catch 316 of 340 real egress events (recall 0.93) while letting only 16 benign files trip the alarm — a 0.5% false-positive rate the SOC can actually staff.

## SEC11 · Account Takeover at Login

At login, every decision has a price tag: lock a real user and you pay a helpdesk ticket; miss an account takeover and you eat fraud plus breach cleanup. Those two dollar figures — not the textbook `0.5` — are what pick your threshold. We score every login with XGBoost, then sweep the decision line against an explicit cost matrix and let dollars choose the operating point. The cost-minimizing threshold falls out at `0.24`, not `0.5` — delivering `recall 0.88 @ FPR 2.0%` on a `PR-AUC 0.895` model, for `$5,773/day` less than the naive line.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, confusion_matrix
from xgboost import XGBClassifier

# --- 1. seeded login world (ATO feature names) ----------------------
X, y = make_classification(
    n_samples=17000, n_features=12, n_informative=8,
    weights=[0.94, 0.06], class_sep=1.6, random_state=8)
COLS = ["geo_distance_km", "device_new", "impossible_travel",
        "hour_of_day", "failed_attempts", "ip_rep", "asn_rare",
        "ua_mismatch", "mfa_fail", "password_reset_recent",
        "session_age", "login_velocity"]
X = pd.DataFrame(X, columns=COLS)

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=8)

# --- 2. model: scale_pos_weight rebalances toward rare ATO class ----
spw = (ytr == 0).sum() / (ytr == 1).sum()
model = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9,
    scale_pos_weight=spw, eval_metric="aucpr",
    early_stopping_rounds=30, random_state=8)
model.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
proba = model.predict_proba(Xte)[:, 1]

# --- 3. honest quality: PR-AUC grades the MODEL, not the line -------
pr_auc = average_precision_score(yte, proba)
print(f"PR-AUC = {pr_auc:.3f}   (model quality -- says nothing about where to operate)")

# --- 4. the cost matrix YOU write down ------------------------------
COST_FP = 8       # lock a real user  -> helpdesk / friction
COST_FN = 400     # missed takeover   -> fraud + breach remediation
DAY_LOGINS = 20000
scale = DAY_LOGINS / len(yte)

# --- 5. sweep the threshold, let dollars pick the operating point ---
grid = np.round(np.arange(0.10, 0.91, 0.02), 2)
rows = []
for t in grid:
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    cost = (fp * COST_FP + fn * COST_FN) * scale
    rows.append((t, tp, fn, fp, tn, cost))
arr = np.array(rows)
best = arr[arr[:, 5].argmin()]
t_star = best[0]

# baseline: the naive 0.5 line, for contrast
pred5 = (proba >= 0.5).astype(int)
tn5, fp5, fn5, tp5 = confusion_matrix(yte, pred5).ravel()
cost5 = (fp5 * COST_FP + fn5 * COST_FN) * scale

# --- 6. report the cost-optimal decision (this is the figure) ------
pred = (proba >= t_star).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
recall = tp / (tp + fn); fpr = fp / (fp + tn)
prec = tp / (tp + fp) if (tp + fp) else 0.0
tnd, fpd, fnd, tpd = (np.array([tn, fp, fn, tp]) * scale).round()
print(f"cost-optimal threshold = {t_star:.2f}")
print(f"  precision={prec:.2f}  recall={recall:.2f}  FPR={fpr*100:.1f}%")
print(f"  daily confusion: TP={tpd:.0f} FN={fnd:.0f} FP={fpd:.0f} TN={tnd:.0f}")
print(f"min expected cost ${best[5]:,.0f}/day  (vs ${cost5:,.0f}/day at the naive 0.50 line)")
print(f"savings vs 0.50 = ${cost5-best[5]:,.0f}/day  ({(1-best[5]/cost5)*100:.0f}% lower)")

# --- 7. hero figure: cost-weighted confusion matrix with $ per cell -
cells = np.array([[tnd*0, fpd*COST_FP],   # TN costs $0 ; FP cost
                  [fnd*COST_FN, tpd*0]])  # FN cost ; TP costs $0
counts = np.array([[tnd, fpd], [fnd, tpd]])
labels = [["TN  (allow good)", "FP  (lock good)"],
          ["FN  (miss ATO)", "TP  (catch ATO)"]]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cells, cmap="Reds")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{labels[i][j]}\n{counts[i,j]:.0f}/day\n${cells[i,j]:,.0f}/day",
                ha="center", va="center", fontsize=10)
ax.set_xticks([0, 1]); ax.set_xticklabels(["pred benign", "pred ATO"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["actual benign", "actual ATO"])
ax.set_title(f"Cost-weighted confusion @ t={t_star:.2f}  (FP=${COST_FP}, FN=${COST_FN})")
fig.colorbar(im, label="$ / day")
fig.tight_layout()
plt.show()

**Output**

```text
PR-AUC = 0.895   (model quality -- says nothing about where to operate)
cost-optimal threshold = 0.24
  precision=0.76  recall=0.88  FPR=2.0%
  daily confusion: TP=1141 FN=153 FP=369 TN=18337
min expected cost $64,125/day  (vs $69,898/day at the naive 0.50 line)
savings vs 0.50 = $5,773/day  (8% lower)
```

The minimum-cost line lands at `0.24`, not `0.5` — because a missed takeover costs 50x a wrongly-locked user, the dollars pull the threshold *down* to catch more attacks. Same model, same scores; only the cost matrix moved the decision, and it bought `$5,773/day` of savings over the naive line.

## SEC12 · Reading the Why — SHAP for Triage

One firing isn't an answer — it's a question: *why this alert?* TreeSHAP decomposes a single phishing score into per-feature contributions, turning a number into a sentence an analyst can act on. For caught alert `url_9947` (score `0.96`), the model says: high `subdomain_count` and an `@` in the URL pushed it up, while a short `url_length` pulled back — net `PR-AUC=0.952`, `precision=0.859`, `recall=0.895`, `FPR@0.5=0.0377`.

In [ ]:
# SEC12 - Reading the Why: SHAP for triage
# Reuses the frozen seed-42 phishing 'house' dataset (SEC02/04/12/13/15/18).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, precision_score, recall_score
from xgboost import XGBClassifier, DMatrix

SEED = 42
FEATURES = ["url_length", "num_dots", "has_https", "subdomain_count",
            "entropy", "age_days", "path_depth", "num_params",
            "has_at_symbol", "is_ip_host", "tld_risk", "redirect_count"]

# 1. frozen 'house' dataset (identical across the phishing episodes)
X, y = make_classification(
    n_samples=12000, n_features=12, n_informative=8,
    weights=[0.8, 0.2], class_sep=1.1, random_state=SEED)
X = pd.DataFrame(X, columns=FEATURES)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)

# 2. the same phishing detector
spw = (ytr == 0).sum() / (ytr == 1).sum()
model = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw,
    eval_metric="aucpr", random_state=SEED)
model.fit(Xtr, ytr)
proba = model.predict_proba(Xte)[:, 1]

# security-honest headline metrics @ threshold 0.5
thr = 0.5
pred = (proba >= thr).astype(int)
pr_auc = average_precision_score(yte, proba)
prec = precision_score(yte, pred)
rec = recall_score(yte, pred)
fp = int(((pred == 1) & (yte == 0)).sum())
tn = int(((pred == 0) & (yte == 0)).sum())
fpr = fp / (fp + tn)
print(f"house model  PR-AUC={pr_auc:.3f}  "
      f"precision={prec:.3f}  recall={rec:.3f}  FPR@0.5={fpr:.4f}")

# 3. pick ONE caught true-positive near the 0.96 headline
tp = (yte == 1) & (proba >= thr)
cand = np.where(tp)[0]
pick = cand[np.argmin(np.abs(proba[cand] - 0.96))]
alert = Xte.iloc[[pick]]
alert_id = f"url_{Xte.index[pick]}"
score = float(proba[pick])
print(f"ALERT {alert_id}  score={score:.2f}  (true phishing, caught)")

# 4. real TreeSHAP. xgboost native pred_contribs gives contributions in
#    MARGIN (log-odds) space; last column is the base value. (Colab also has
#    the `shap` package: shap.TreeExplainer(model)(alert) gives the same
#    decomposition with a built-in waterfall plot.)
booster = model.get_booster()
dm = DMatrix(alert.values, feature_names=FEATURES)
contribs = booster.predict(dm, pred_contribs=True)[0]   # len = n_feat + 1
base_margin = float(contribs[-1])
feat_contribs = pd.Series(contribs[:-1], index=FEATURES)
top = feat_contribs.reindex(feat_contribs.abs().sort_values(ascending=False).index)
top3 = top.head(3)
margin_total = base_margin + feat_contribs.sum()
prob_from_margin = 1.0 / (1.0 + np.exp(-margin_total))

print(f"base margin (log-odds) = {base_margin:+.3f}")
print("top pushers (log-odds): " +
      ", ".join(f"{k} {v:+.2f}" for k, v in top3.items()))
print(f"sum(base + contribs) = {margin_total:+.3f} log-odds "
      f"-> sigmoid = {prob_from_margin:.3f}  (= the score)")

# additivity is a hard check
assert abs(prob_from_margin - score) < 1e-4, (prob_from_margin, score)

# 5. waterfall: the actionable sentence, drawn
order = top.head(8).index[::-1]
vals = top.loc[order].values
colors = ["#ff6b6b" if v > 0 else "#7cc1ff" for v in vals]
plt.figure(figsize=(8, 5))
plt.barh(range(len(order)), vals, color=colors)
plt.yticks(range(len(order)), order)
plt.axvline(0, color="#444", lw=1)
plt.title(f"Why {alert_id} fired  (score {score:.2f})")
plt.xlabel("SHAP contribution to log-odds")
plt.tight_layout()
plt.show()

**Output**

```text
house model  PR-AUC=0.952  precision=0.859  recall=0.895  FPR@0.5=0.0377
ALERT url_9947  score=0.96  (true phishing, caught)
base margin (log-odds) = +0.087
top pushers (log-odds): subdomain_count +0.98, has_at_symbol +0.67, url_length -0.64
sum(base + contribs) = +3.176 log-odds -> sigmoid = 0.960  (= the score)
```

The waterfall reads as a sentence: *this alert fired because the URL has an unusual number of subdomains and an `@` symbol — classic obfuscation — even though its short length argued mildly against it.* That's the line a SOC analyst pastes into the ticket.

## SEC13 · The False-Positive Budget

Your SOC can only chase so many alerts a day. The trick isn't a better model — it's picking the decision threshold *on purpose*. Sweep the threshold, draw your false-positive budget as a line, and read off the recall you can actually afford. On the shared seed-42 phishing dataset, the hidden default 0.50 cutoff floods the queue with `~1,486 alerts/day` at `recall 0.891`; holding a `300/day` FP budget instead picks `threshold 0.805` for `recall 0.761 · precision 0.965 · FPR 0.698% (~278 alerts/day)`.

In [ ]:
# SEC13 — The False-Positive Budget: sweep the threshold, draw your FP budget, read off the recall
import numpy as np, pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

SEED = 42

# --- 1. Canonical seed-42 phishing 'house' dataset (shared across episodes) ---
X, y = make_classification(
    n_samples=12000, n_features=12, n_informative=8,
    weights=[0.8, 0.2], class_sep=1.1, random_state=SEED)
cols = ["url_length", "num_dots", "has_https", "subdomain_count",
        "entropy", "age_days", "num_hyphens", "path_depth",
        "num_params", "num_digits", "tld_rank", "redirect_count"]
X = pd.DataFrame(X, columns=cols)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)

# --- 2. Train the detector; use SCORES not .predict() (the hidden 0.5 cutoff) ---
spw = (ytr == 0).sum() / (ytr == 1).sum()
clf = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, eval_metric="aucpr",
    scale_pos_weight=spw, random_state=SEED)
clf.fit(Xtr, ytr)
scores = clf.predict_proba(Xte)[:, 1]
prauc = average_precision_score(yte, scores)

# --- 3. Sweep threshold; convert FPR -> daily alert count ---
DAILY_URLS = 50_000
PREVALENCE = yte.mean()
benign_per_day = DAILY_URLS * (1 - PREVALENCE)
thresholds = np.round(np.arange(0.05, 0.991, 0.005), 3)

rows = []
for t in thresholds:
    pred = scores >= t
    tp = int((pred & (yte == 1)).sum())
    fp = int((pred & (yte == 0)).sum())
    fn = int((~pred & (yte == 1)).sum())
    tn = int((~pred & (yte == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 1.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    alerts = fpr * benign_per_day
    rows.append((t, precision, recall, fpr, alerts))
sweep = pd.DataFrame(rows, columns=["thr", "precision", "recall", "fpr", "alerts"])

# --- 4. Apply the FP budget: highest recall while alerts/day <= budget ---
BUDGET = 300
affordable = sweep[sweep["alerts"] <= BUDGET]
chosen = affordable.loc[affordable["recall"].idxmax()]
base = sweep.iloc[(sweep["thr"] - 0.5).abs().idxmin()]

print(f"PR-AUC (test): {prauc:.3f}")
print(f"prevalence: {PREVALENCE*100:.1f}% phishing | benign/day ~ {benign_per_day:,.0f}")
print()
print(f"DEFAULT 0.50 cutoff -> recall {base.recall:.3f} | precision {base.precision:.3f} | "
      f"FPR {base.fpr*100:.3f}% | ~{base.alerts:,.0f} alerts/day")
print()
print(f"BUDGET-PICKED thr={chosen.thr:.3f} -> recall {chosen.recall:.3f} | "
      f"precision {chosen.precision:.3f} | FPR {chosen.fpr*100:.3f}% | "
      f"~{chosen.alerts:.0f} alerts/day (budget {BUDGET})")

# --- 5. Plot the sweep with the amber budget line + chosen threshold ---
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(sweep.thr, sweep.precision, color="#56D364", lw=2.2, label="precision")
ax.plot(sweep.thr, sweep.recall, color="#FF5C5C", lw=2.2, label="recall")
ax2 = ax.twinx()
ax2.plot(sweep.thr, sweep.alerts, color="#79828F", lw=1.8, ls="--", label="alerts/day")
ax2.axhline(BUDGET, color="#FFB454", lw=2.0, label=f"FP budget {BUDGET}/day")
ax.axvline(chosen.thr, color="#FFB454", lw=1.6, ls=":")
ax.set_xlabel("decision threshold"); ax.set_ylabel("precision / recall")
ax2.set_ylabel("false alerts per day"); ax.set_title("Threshold vs the FP budget")
ax.legend(loc="center left"); ax2.legend(loc="upper right")
fig.tight_layout(); plt.show()

**Output**

```text
PR-AUC (test): 0.948
prevalence: 20.4% phishing | benign/day ~ 39,819

DEFAULT 0.50 cutoff -> recall 0.891 | precision 0.859 | FPR 3.732% | ~1,486 alerts/day

BUDGET-PICKED thr=0.805 -> recall 0.761 | precision 0.965 | FPR 0.698% | ~278 alerts/day (budget 300)
```

The default 0.50 cutoff buys high recall but dumps `~1,486` false alerts a day on your analysts. Hold a `300/day` budget and the sweep hands you `threshold 0.805`: recall drops to `0.761`, but precision jumps to `0.965` and the queue shrinks to `~278/day` — a load your team can actually clear.

## SEC14 · Encoding Threat-Model Priors

A model that scores high on PR-AUC can still hide an attacker soft spot: somewhere on the `failed_auth` curve, *more* failed logins make the model predict *less* risk. That's a free-form tree fit picking up noise. `monotone_constraints` lets us bake the threat model straight into the booster — "more-suspicious can only raise risk" — and here it costs nothing: `PR-AUC 0.990 constrained vs 0.989 free` · `recall 0.97 @ FPR 1% (vs 0.95)` · monotone response non-decreasing by construction.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # Colab: drop this line if you want inline plots
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from sklearn.inspection import partial_dependence
import xgboost as xgb

# --- seeded abuse world: 9 behavioral features, ~12% abuse ----------------
# Each feature carries signal in the direction the threat model expects, so a
# monotone prior is HONEST (not forced onto noise).
FEATURES = ["failed_auth", "req_rate", "entropy", "ip_rep",
            "account_age_days", "distinct_ua", "captcha_fails",
            "geo_risk", "token_reuse"]
rng = np.random.default_rng(19)
n = 14000
z = rng.normal(0, 1, n)
y = (z > np.quantile(z, 0.88)).astype(int)          # ~12% abuse

def feat(strength, sign=1):
    return rng.normal(0, 1, n) + sign * strength * (y - 0.12)

X = np.column_stack([
    feat(2.2),                              # failed_auth   ^ risk
    rng.normal(0, 1, n) + 1.2 * (y - .12),  # req_rate      (no constraint)
    feat(1.8),                              # entropy       ^ risk
    rng.normal(0, 1, n) + 1.0 * (y - .12),  # ip_rep        (no constraint)
    feat(1.5, sign=-1),                     # account_age   v risk
    rng.normal(0, 1, n),                    # distinct_ua   noise
    feat(1.6),                              # captcha_fails ^ risk
    feat(1.4),                              # geo_risk      ^ risk
    rng.normal(0, 1, n) + 0.8 * (y - .12),  # token_reuse   (no constraint)
])
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=19)

# --- threat model as a vector (one entry per feature) ---------------------
# 1 = risk may only RISE,  -1 = may only FALL,  0 = no constraint
mono = [1, 0, 1, 0, -1, 0, 1, 1, 0]   # failed_auth^ entropy^ captcha^ geo^ age v

spw = (y == 0).sum() / (y == 1).sum()
common = dict(n_estimators=400, max_depth=4, learning_rate=0.05,
              subsample=0.9, colsample_bytree=0.9, eval_metric="aucpr",
              scale_pos_weight=spw, random_state=19, n_jobs=4)

free = xgb.XGBClassifier(**common)
free.fit(Xtr, ytr)

mono_model = xgb.XGBClassifier(monotone_constraints=tuple(mono), **common)
mono_model.fit(Xtr, ytr)

# --- security-honest scoring: PR-AUC + recall at a fixed false-alarm rate --
def recall_at_fpr(yt, score, target_fpr=0.01):
    neg = score[yt == 0]
    thr = np.quantile(neg, 1 - target_fpr)        # 1% of benign get flagged
    return (score[yt == 1] >= thr).mean()         # share of abuse we catch

print("=== Encoding Threat-Model Priors: monotone_constraints ===")
print(f"abuse base rate (test): {yte.mean():.3f}   scale_pos_weight={spw:.2f}\n")
for name, m in [("free", free), ("monotone", mono_model)]:
    p = m.predict_proba(Xte)[:, 1]
    ap = average_precision_score(yte, p)
    rec = recall_at_fpr(yte, p, 0.01)
    print(f"{name:9s}  PR-AUC={ap:.3f}   recall@FPR1%={rec:.2f}")

# --- hero figure: partial dependence of failed_auth -----------------------
idx = FEATURES.index("failed_auth")   # column 0
grid = {"grid_resolution": 30}
pd_free = partial_dependence(free, Xte, [idx], **grid)
pd_mono = partial_dependence(mono_model, Xte, [idx], **grid)
xs = pd_free["grid_values"][0]
yf = pd_free["average"][0]   # free: wiggles, can DIP (a soft spot for attackers)
ym = pd_mono["average"][0]   # monotone: non-decreasing by construction

dip = np.diff(yf).min()
print(f"\nfree failed_auth response:     min step = {dip:+.4f}  "
      f"({'dips' if dip < -1e-9 else 'no dip'})")
print(f"monotone failed_auth response: min step = {np.diff(ym).min():+.4f}")
assert np.all(np.diff(ym) >= -1e-9), "constraint broke - monotone must not fall"
print("monotone curve is non-decreasing by construction  [OK]")

plt.figure(figsize=(7, 4.5))
plt.plot(xs, yf, "-o", ms=3, color="#ff6b6b", label="free (dips = attacker soft spot)")
plt.plot(xs, ym, "-o", ms=3, color="#9fd97a", label="monotone_constraints=1")
plt.xlabel("failed_auth")
plt.ylabel("partial dependence (risk)")
plt.title("Threat-model prior: more failed_auth -> only higher risk")
plt.legend()
plt.tight_layout()
plt.show()

**Output**

```text
=== Encoding Threat-Model Priors: monotone_constraints ===
abuse base rate (test): 0.120   scale_pos_weight=7.33

free       PR-AUC=0.989   recall@FPR1%=0.95
monotone   PR-AUC=0.990   recall@FPR1%=0.97

free failed_auth response:     min step = -0.0006  (dips)
monotone failed_auth response: min step = +0.0000
monotone curve is non-decreasing by construction  [OK]
```

## SEC15 · Calibrating the Risk Score

Boosted trees rank threats brilliantly but their raw scores are **not probabilities** — push the model hard enough and it pins scores to the rails. Before you wire a model into *auto-containment*, you have to know that a 0.9 really means 90%. We freeze the trees and fit an isotonic calibrator on a held-out split, then read the receipts: Brier, a 10-bin ECE, and PR-AUC.

Real run on the seed-42 phishing dataset: `Brier 0.038 → 0.037` · `ECE 0.031 → 0.012` · `PR-AUC 0.946 → 0.937 (ranking preserved)`.

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import (brier_score_loss, average_precision_score,
                             precision_score, recall_score)
import matplotlib.pyplot as plt

COLS = ["url_length", "num_dots", "has_https", "subdomain_count",
        "entropy", "age_days", "num_hyphens", "has_at_symbol",
        "path_depth", "num_digits", "brand_in_subdomain", "redirect_count"]

# recurring seed-42 phishing 'house' dataset (same as the other episodes)
X, y = make_classification(n_samples=12000, n_features=12, n_informative=8,
                           weights=[0.8, 0.2], class_sep=1.1, random_state=42)

# three-way split: train trees / fit calibrator / judge both
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=42)
X_cal, X_te, y_cal, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

# A deep, hard-pushed booster -- the realistic cause of miscalibration in
# the field. Many deep, unregularized trees drive scores to the rails, so
# the raw probabilities are over-confident even though ranking is excellent.
clf = xgb.XGBClassifier(
    n_estimators=900, max_depth=12, learning_rate=0.1,
    min_child_weight=1, reg_lambda=0,
    eval_metric="logloss", random_state=42, n_jobs=4)
clf.fit(X_tr, y_tr, verbose=False)

# raw boosted scores -- NOT probabilities
p_raw = clf.predict_proba(X_te)[:, 1]

# isotonic calibration on the held-out split (freeze the trees first)
cal = CalibratedClassifierCV(FrozenEstimator(clf), method="isotonic")
cal.fit(X_cal, y_cal)
p_cal = cal.predict_proba(X_te)[:, 1]


def ece(y_true, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    err = 0.0
    for i in range(bins):
        hi = p < edges[i + 1] if i < bins - 1 else p <= edges[i + 1]
        m = (p >= edges[i]) & hi
        if m.sum() == 0:
            continue
        err += (m.sum() / len(p)) * abs(y_true[m].mean() - p[m].mean())
    return err


b_raw, b_cal = brier_score_loss(y_te, p_raw), brier_score_loss(y_te, p_cal)
e_raw, e_cal = ece(y_te, p_raw), ece(y_te, p_cal)
ap_raw = average_precision_score(y_te, p_raw)
ap_cal = average_precision_score(y_te, p_cal)

# security-honest operating point: auto-contain anything scored >= 0.5
thr = 0.5
pred_raw = (p_raw >= thr).astype(int)
pred_cal = (p_cal >= thr).astype(int)
neg = y_te == 0
fpr_raw = pred_raw[neg].mean()
fpr_cal = pred_cal[neg].mean()

print("=== Calibrating the Risk Score (SEC15) ===")
print(f"test set: {len(y_te)} urls, {int(y_te.sum())} malicious "
      f"({y_te.mean():.1%})")
print()
print(f"Brier   {b_raw:.3f} -> {b_cal:.3f}")
print(f"ECE     {e_raw:.3f} -> {e_cal:.3f}")
print(f"PR-AUC  {ap_raw:.3f} -> {ap_cal:.3f}  (ranking preserved)")
print()
print(f"auto-contain @ score>=0.50")
print(f"  precision  {precision_score(y_te, pred_raw):.3f} -> "
      f"{precision_score(y_te, pred_cal):.3f}")
print(f"  recall     {recall_score(y_te, pred_raw):.3f} -> "
      f"{recall_score(y_te, pred_cal):.3f}")
print(f"  FPR        {fpr_raw:.3f} -> {fpr_cal:.3f}")

# hero figure: reliability curve, raw vs isotonic, Brier annotated
ft_raw, mp_raw = calibration_curve(y_te, p_raw, n_bins=10)
ft_cal, mp_cal = calibration_curve(y_te, p_cal, n_bins=10)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "--", color="#7d8693", label="perfect")
plt.plot(mp_raw, ft_raw, "o-", color="#FF5C5C",
         label=f"raw XGBoost (Brier {b_raw:.3f})")
plt.plot(mp_cal, ft_cal, "o-", color="#56D364",
         label=f"isotonic (Brier {b_cal:.3f})")
plt.xlabel("mean predicted probability")
plt.ylabel("observed fraction positive")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Output**

```text
=== Calibrating the Risk Score (SEC15) ===
test set: 2400 urls, 488 malicious (20.3%)

Brier   0.038 -> 0.037
ECE     0.031 -> 0.012
PR-AUC  0.946 -> 0.937  (ranking preserved)

auto-contain @ score>=0.50
  precision  0.940 -> 0.885
  recall     0.840 -> 0.867
  FPR        0.014 -> 0.029
```

Isotonic cuts calibration error (ECE) by more than half — `0.031 → 0.012` — while PR-AUC barely moves (`0.946 → 0.937`): the model's *ranking* was always good, only its *probabilities* lied. After calibration a 0.5 score is closer to a true coin-flip, so the auto-contain decision is honest: recall climbs (0.840 → 0.867) at a modest, predictable FPR cost. Calibrate before you let a score trigger containment.

## SEC16 · Concept Drift & Retraining

A phishing detector is never *done*. We train one on month 0, freeze it, then watch attackers slowly adapt over eight monthly batches — tracking rolling PR-AUC and retraining the moment it breaches a floor. The frozen model rots while the retrain-on-breach champion snaps back. Frozen drift: `0.984 → 0.875` PR-AUC (lost `0.109` over 8 months) · retrains at `month 6` · floor `0.93`.

In [ ]:
"""SEC16 · Concept Drift & Retraining — XGBoost for Cyber Defense.
Frozen phishing detector watched over 8 monthly batches as attackers adapt;
rolling PR-AUC triggers a retrain on a floor breach. Self-contained, no downloads."""
import numpy as np, pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import average_precision_score

FEATS = ["url_length","num_dots","has_https","subdomain_count","entropy","age_days"]
FLOOR = 0.93                                    # retrain-trigger PR-AUC floor

def make_month(month, half):
    # half='train'|'eval' -> disjoint resampling: a retrained model is never
    # scored on data it trained on (honest forward validation).
    sep = 2.2 - 0.06 * month                    # attackers blend into benign traffic
    X, y = make_classification(n_samples=5000, n_features=6, n_informative=4,
        n_redundant=1, weights=[0.82, 0.18], class_sep=sep,
        flip_y=0.012, random_state=42)          # geometry FIXED across months
    seed = 100 + month * 2 + (0 if half == "train" else 1)
    rng = np.random.default_rng(seed)           # disjoint sampling noise per half
    X += rng.normal(0, 0.045, X.shape)
    d = 0.14 * month                            # covariate shift on the attack class
    for c in range(5):                          # attackers nudge their features
        X[y == 1, c] += d * (0.6 if c % 2 else 1.0)
    return pd.DataFrame(X, columns=FEATS), y

PARAMS = dict(n_estimators=300, max_depth=5, learning_rate=0.08, subsample=0.9,
    colsample_bytree=0.9, eval_metric="aucpr", random_state=0, n_jobs=2)

# frozen "house" model trained on month 0
Xtr, ytr = make_month(0, "train")
Xho, yho = make_month(0, "eval")
frozen = xgb.XGBClassifier(**PARAMS).fit(Xtr, ytr)
base = average_precision_score(yho, frozen.predict_proba(Xho)[:, 1])

# roll forward 8 months: frozen vs retrain-on-breach champion
months, frozen_pr, retrain_pr, retrains = [], [], [], []
champion = frozen
for m in range(1, 9):
    Xe, ye = make_month(m, "eval")              # this month's monitoring traffic
    pf = average_precision_score(ye, frozen.predict_proba(Xe)[:, 1])
    pr = average_precision_score(ye, champion.predict_proba(Xe)[:, 1])
    months.append(m); frozen_pr.append(round(pf, 3)); retrain_pr.append(round(pr, 3))
    if pr < FLOOR:                              # breach -> retrain on fresh month
        Xt, yt = make_month(m, "train")         # disjoint from the eval half above
        champion = xgb.XGBClassifier(**PARAMS).fit(Xt, yt)
        retrains.append(m)

print(f"Frozen model trained on month 0  ·  PR-AUC = {base:.3f}")
print(f"{'month':>6}{'frozen':>9}{'retrained':>11}")
for m, f, r in zip(months, frozen_pr, retrain_pr):
    flag = "  <- BREACH: retrain" if r < FLOOR else ""
    print(f"{'M'+str(m):>6}{f:>9.3f}{r:>11.3f}{flag}")
print(f"Frozen drift: {frozen_pr[0]:.3f} -> {frozen_pr[-1]:.3f}  "
      f"(lost {frozen_pr[0]-frozen_pr[-1]:.3f} PR-AUC over 8 months)")
print(f"Retrained holds >= {min(retrain_pr):.3f} (floor {FLOOR}) · retrains at month(s) {retrains}")

# hero figure: monthly PR-AUC, frozen vs retrained, with trigger floor
plt.figure(figsize=(8,5))
plt.plot(months, frozen_pr, "o-", color="#FF5C5C", lw=2.5, label="frozen model")
plt.plot(months, retrain_pr, "o-", color="#56D364", lw=2.5, label="retrain on breach")
plt.axhline(FLOOR, color="#FFB454", ls="--", lw=2, label=f"retrain floor {FLOOR}")
plt.xlabel("month"); plt.ylabel("PR-AUC"); plt.ylim(0.84, 1.0)
plt.title("A frozen detector rots; rolling PR-AUC tells you when to retrain")
plt.legend(); plt.tight_layout(); plt.show()

**Output**

```text
Frozen model trained on month 0  ·  PR-AUC = 0.991
 month   frozen  retrained
    M1    0.984      0.984
    M2    0.976      0.976
    M3    0.968      0.968
    M4    0.959      0.959
    M5    0.942      0.942
    M6    0.924      0.924  <- BREACH: retrain
    M7    0.902      1.000
    M8    0.875      1.000
Frozen drift: 0.984 -> 0.875  (lost 0.109 PR-AUC over 8 months)
Retrained holds >= 0.924 (floor 0.93) · retrains at month(s) [6]
```

## SEC17 · Adversarial Evasion

Train the house phishing detector, then attack it. An adversary nudges only malicious URLs along four cheap-to-spoof features (`url_length`, `has_https`, `num_dots`, `entropy`) toward the benign average, bounded by a per-feature sigma budget. We sweep the budget and watch recall collapse — then retrain with bounded-perturbed copies of the malicious rows mixed in. Real result: baseline recall halves under full budget (`0.82 -> 0.49`), the adversarially-trained model holds `0.76`, and clean PR-AUC barely moves (`0.947 -> 0.939`).

In [ ]:
# 17_adversarial_evasion.py - measure evasion robustness, then harden
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, recall_score
import xgboost as xgb

SEED = 42
np.random.seed(SEED)

# --- the recurring seed-42 phishing 'house' dataset (identical across episodes) ---
FEATURES = ["url_length", "num_dots", "subdomain_count", "has_https",
            "entropy", "age_days", "num_hyphens", "has_at_symbol",
            "path_depth", "num_digits", "brand_in_subdomain", "redirect_count"]
X, y = make_classification(
    n_samples=12000, n_features=12, n_informative=8,
    weights=[0.8, 0.2], class_sep=1.1, random_state=SEED)
df = pd.DataFrame(X, columns=FEATURES)
Xtr, Xte, ytr, yte = train_test_split(
    df, y, test_size=0.25, stratify=y, random_state=SEED)

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.08,
              subsample=0.9, colsample_bytree=0.9,
              eval_metric="aucpr", random_state=SEED)

# --- baseline model (the house phishing detector) ---
base = xgb.XGBClassifier(**PARAMS).fit(Xtr, ytr)

# --- threat model: 4 cheap-to-spoof, high-leverage features ---
EVADE = ["url_length", "has_https", "num_dots", "entropy"]
sigma = Xtr[EVADE].std().values            # bounds the budget in real units
benign_mean = Xtr[ytr == 0][EVADE].mean().values

def perturb(Xpos, budget, scale=0.9):
    """nudge malicious rows along EVADE toward benign, within budget*sigma."""
    Z = Xpos.copy()
    cur = Z[EVADE].values
    direction = np.sign(benign_mean - cur)         # move toward 'benign'
    Z[EVADE] = cur + direction * budget * sigma * scale
    return Z

def recall_at_budget(model, budget, thr=0.5):
    Xe = Xte.copy()
    pos = (yte == 1)                               # attack only malicious rows
    Xe.loc[pos, :] = perturb(Xte[pos], budget)
    p = model.predict_proba(Xe)[:, 1]
    return recall_score(yte, (p >= thr).astype(int))

# --- adversarial training: augment with bounded-perturbed malicious rows ---
pos_tr = Xtr[ytr == 1]
augs = [perturb(pos_tr, budget=b) for b in (0.3, 0.6)]
Xtr_h = pd.concat([Xtr, *augs], ignore_index=True)
ytr_h = np.concatenate([ytr] + [np.ones(len(a), dtype=int) for a in augs])
hard = xgb.XGBClassifier(**PARAMS).fit(Xtr_h, ytr_h)

# --- clean PR-AUC for both models ---
prauc_base = average_precision_score(yte, base.predict_proba(Xte)[:, 1])
prauc_hard = average_precision_score(yte, hard.predict_proba(Xte)[:, 1])

# --- recall-vs-evasion-budget sweep ---
budgets = [0.0, 0.25, 0.5, 0.75, 1.0]
rec_base = [recall_at_budget(base, b) for b in budgets]
rec_hard = [recall_at_budget(hard, b) for b in budgets]

print(f"clean PR-AUC : baseline {prauc_base:.3f}  hardened {prauc_hard:.3f}")
print("budget(sigma) | recall_base | recall_hard")
for b, rb, rh in zip(budgets, rec_base, rec_hard):
    print(f"   {b:.2f}       |    {rb:.2f}     |    {rh:.2f}")
print(f"FULL-BUDGET DROP: baseline {rec_base[0]:.2f} -> {rec_base[-1]:.2f}"
      f" | hardened holds {rec_hard[-1]:.2f}")

# --- hero figure: recall vs evasion budget ---
plt.figure(figsize=(7, 4.5))
plt.plot(budgets, rec_base, "o-", color="#ff6b6b", lw=2.5, label="baseline")
plt.plot(budgets, rec_hard, "s-", color="#9fd97a", lw=2.5, label="adversarially trained")
plt.xlabel("evasion budget (x sigma)"); plt.ylabel("recall on malicious")
plt.title("Recall under bounded-perturbation evasion")
plt.ylim(0, 1); plt.legend(); plt.grid(alpha=0.2)
plt.tight_layout(); plt.show()

**Output**

```text
clean PR-AUC : baseline 0.947  hardened 0.939
budget(sigma) | recall_base | recall_hard
   0.00       |    0.82     |    0.90
   0.25       |    0.75     |    0.90
   0.50       |    0.67     |    0.91
   0.75       |    0.57     |    0.86
   1.00       |    0.49     |    0.76
FULL-BUDGET DROP: baseline 0.82 -> 0.49 | hardened holds 0.76
```

## SEC18 · The SOC Scoring Pipeline

The capstone: take the Ep02 phishing booster and wrap it into a *shippable* SOC detector. Not one number, but a pipeline — rebalance the classes, constrain risk-only features to be monotone, early-stop on AUCPR, calibrate the score into an honest probability, then split it into three operating tiers (auto-block / triage / pass) and explain every auto-block with TreeSHAP. A production detector is a *calibrated, constrained, thresholded, explained* pipeline, not a `.fit()` call.

Real headline on the frozen seed-42 phishing set: `PR-AUC 0.934` · `auto-block precision 0.98 @ score≥0.93` · `block+triage recall 0.86` · `FPR 2.4%` · `~582 alerts/day escalated`.

In [ ]:
# CAPSTONE: the Ep02 phishing model, wrapped into a shippable tiered detector.
# rebalance -> constrain -> early-stop -> calibrate -> threshold -> explain.
import numpy as np, shap
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import average_precision_score, precision_recall_curve
from xgboost import XGBClassifier

RNG = 42
FEATS = ["url_length", "num_dots", "has_https", "subdomain_count",
         "url_entropy", "domain_age_days", "num_hyphens", "path_depth",
         "num_digits", "is_ip_host", "tld_risk", "redirect_count"]

# --- the recurring Ep02 phishing "house" dataset, frozen under seed 42 ---
X, y = make_classification(n_samples=12000, n_features=12, n_informative=8,
                           weights=[0.8, 0.2], class_sep=1.1, random_state=RNG)
# train | calib-holdout | test  (calibration must never see training rows)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.40,
                                            stratify=y, random_state=RNG)
X_cal, X_te, y_cal, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50,
                                            stratify=y_tmp, random_state=RNG)

# --- one constrained, balanced, early-stopped booster --------------
spw = (y_tr == 0).sum() / (y_tr == 1).sum()      # imbalance weight (Ep03)
# subdomain_count & url_entropy: risk-only monotone-increasing (Ep14)
mono = tuple(1 if f in ("subdomain_count", "url_entropy") else 0 for f in FEATS)
clf = XGBClassifier(
    n_estimators=2000, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw, monotone_constraints=mono,
    eval_metric="aucpr", early_stopping_rounds=30,   # stop smart (Ep07)
    random_state=RNG, n_jobs=4)
clf.fit(X_tr, y_tr, eval_set=[(X_cal, y_cal)], verbose=False)
print(f"early-stopped at {clf.best_iteration + 1} trees")

# --- calibrate so the score is an honest probability (Ep15) ---------
# sklearn dropped cv="prefit" -> wrap the fitted booster in FrozenEstimator.
cal = CalibratedClassifierCV(FrozenEstimator(clf), method="isotonic")
cal.fit(X_cal, y_cal)
p = cal.predict_proba(X_te)[:, 1]                 # honest P(phishing)
prauc = average_precision_score(y_te, p)
print(f"PR-AUC (test) = {prauc:.3f}")

# --- threshold sweep -> two operating points, three tiers (Ep13) ----
T_BLOCK, T_TRIAGE = 0.93, 0.40
block  = p >= T_BLOCK
triage = (p >= T_TRIAGE) & ~block
passed = p < T_TRIAGE
def stats(mask):
    n = int(mask.sum()); tp = int((y_te[mask] == 1).sum())
    return n, tp, (tp / n if n else 0.0)
tier_rows = []
for name, m in [("block", block), ("triage", triage), ("pass", passed)]:
    n, tp, prec = stats(m); tier_rows.append((name, n, tp, prec))
    print(f"{name:6s} n={n:4d}  phish={tp:4d}  precision={prec:.2f}")
caught = (block | triage) & (y_te == 1)
fp     = (block | triage) & (y_te == 0)
recall = caught.sum() / (y_te == 1).sum()
fpr    = fp.sum() / (y_te == 0).sum()
triage_recall = (triage & (y_te == 1)).sum() / (y_te == 1).sum()
print(f"auto-block precision (score>={T_BLOCK}) = {tier_rows[0][3]:.2f}")
print(f"recall(block+triage) = {recall:.2f}")
print(f"triage-tier recall   = {triage_recall:.2f}")
print(f"FPR = {fpr:.3f}")
daily_emails = 3000; scale = daily_emails / len(y_te)
print(f"escalated/day (~{daily_emails} screened) = {(block.sum()+triage.sum())*scale:.0f}")

# --- the reason: SHAP on the auto-block tier (Ep12) -----------------
Xb = X_te[block]
sv = shap.TreeExplainer(clf).shap_values(Xb)
imp = np.abs(sv).mean(axis=0)
top3 = np.argsort(imp)[::-1][:3]
print("auto-block reasons:", [f"{FEATS[i]}={imp[i]:.2f}" for i in top3])
print("PIPELINE READY -> route block / triage / pass")

# --- hero figure: PR curve + per-tier counts + tier x label matrix --
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
prec_c, rec_c, _ = precision_recall_curve(y_te, p)
ax[0].plot(rec_c, prec_c, color="#7cc1ff")
ax[0].set_title(f"PR curve (AUC={prauc:.3f})"); ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision")
names = [r[0] for r in tier_rows]; counts = [r[1] for r in tier_rows]
ax[1].bar(names, counts, color=["#ff6b6b", "#ffd166", "#9fd97a"]); ax[1].set_title("per-tier counts")
mat = np.array([[r[2], r[1]-r[2]] for r in tier_rows])
ax[2].imshow(mat, cmap="magma", aspect="auto")
ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(["phish", "benign"])
ax[2].set_yticks(range(3)); ax[2].set_yticklabels(names); ax[2].set_title("tier x label matrix")
for i in range(3):
    for j in range(2):
        ax[2].text(j, i, int(mat[i, j]), ha="center", va="center", color="w")
plt.tight_layout(); plt.show()

**Output**

```text
early-stopped at 593 trees
PR-AUC (test) = 0.934
block  n= 343  phish= 335  precision=0.98
triage n= 123  phish=  86  precision=0.70
pass   n=1934  phish=  67  precision=0.03
auto-block precision (score>=0.93) = 0.98
recall(block+triage) = 0.86
triage-tier recall   = 0.18
FPR = 0.024
escalated/day (~3000 screened) = 582
auto-block reasons: ['url_length=2.50', 'num_hyphens=0.88', 'num_digits=0.86']
PIPELINE READY -> route block / triage / pass
```

The auto-block tier fires on only 343 of 2400 alerts yet is 98% precise — those 335 confirmed phish get killed without a human. The triage tier hands 123 borderline cases to an analyst at 70% precision. The pass tier (81% of volume) is left alone at 3% residual phish. Across the screened day that's ~582 escalations instead of 2400 — and every auto-block ships with its top-3 TreeSHAP reasons attached, so the analyst sees *why*, not just *what*.